In [2]:
import nb_setup
import torch
import random
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from src.data import load_tokenised_data
from src.answerability.evaluation import evaluate_answerability_predictions
from src.answerability.gru_dataset import PretrainedAnswerabilityDataset, make_collate_fn
from src.answerability.logistic_regression import (
    train_logistic_regression,
    predict_logistic_regression,
    logistic_feature_weights,
)
from src.answerability.gru_model import (
    PretrainedGRUClassifier,
    train_one_epoch_answerability,
    predict_answerability,
    evaluate_macro_f1
)

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

In [4]:
df_train_tokenised, df_val_tokenised = load_tokenised_data()

In [5]:
lr_model, lr_vectorizer = train_logistic_regression(df_train_tokenised)

In [6]:
lr_pred, lr_prob = predict_logistic_regression(lr_model, lr_vectorizer, df_val_tokenised)

In [7]:
y_val = df_val_tokenised["answerable"].astype(int).to_numpy()
val_langs = df_val_tokenised["lang"].to_numpy()

In [8]:
lr_results = evaluate_answerability_predictions(y_val, lr_pred, val_langs, "Logistic Regression").round(4)
lr_results

,Model,Language,N,Macro F1,Unanswerable Precision,Unanswerable Recall,Unanswerable F1,Answerable Precision,Answerable Recall,Answerable F1
0,Logistic Regression,Overall,1155,0.7236,0.8108,0.3659,0.5042,0.9038,0.9859,0.9431
1,Logistic Regression,Arabic,415,0.9459,0.8889,0.9231,0.9057,0.9889,0.9835,0.9862
2,Logistic Regression,Korean,356,0.4863,0.0000,0.0000,0.0000,0.9466,1.0000,0.9726
3,Logistic Regression,Telugu,384,0.5383,0.6000,0.1290,0.2124,0.7775,0.9725,0.8641


In [9]:
feature_weights = logistic_feature_weights(lr_model, lr_vectorizer)

In [10]:
feature_weights.head(15)

,feature,weight
0,q_word=هل,-6.758095
1,q_word=수,-1.216855
2,q_word=있을까,-0.925031
3,q_word=있는가,-0.923192
4,c_word=center,-0.884653
5,q_word=దేశం,-0.881877
6,c_word=armed,-0.799558
7,c_word=elections,-0.761923
8,q_word=ريشتر,-0.746954
9,q_word=الجيولوجي,-0.746954


In [11]:
feature_weights.tail(15)

,feature,weight
56966,q_word=무엇인가요,0.545076
56967,c_word=official,0.564311
56968,q_word=أين,0.567153
56969,c_word=following,0.586641
56970,q_word=어디인가,0.628492
56971,c_word=so,0.631174
56972,c_word=.,0.661460
56973,q_word=ఎన్ని,0.672766
56974,q_word=كم,0.713128
56975,context_has_digit,0.758185


In [12]:
majority_class = df_train_tokenised["answerable"].astype(int).mode()[0]
majority_pred = np.full(len(y_val), majority_class)

majority_results = evaluate_answerability_predictions(y_val, majority_pred, val_langs, "Majority Baseline").round(4)
majority_results

,Model,Language,N,Macro F1,Unanswerable Precision,Unanswerable Recall,Unanswerable F1,Answerable Precision,Answerable Recall,Answerable F1
0,Majority Baseline,Overall,1155,0.4618,0.0,0.0,0.0,0.8580,1.0,0.9236
1,Majority Baseline,Arabic,415,0.4666,0.0,0.0,0.0,0.8747,1.0,0.9332
2,Majority Baseline,Korean,356,0.4863,0.0,0.0,0.0,0.9466,1.0,0.9726
3,Majority Baseline,Telugu,384,0.4311,0.0,0.0,0.0,0.7578,1.0,0.8622


In [13]:
MODEL_NAME = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
xlmr = AutoModel.from_pretrained(MODEL_NAME)

pretrained_embeddings = (
    xlmr
    .get_input_embeddings()
    .weight
    .detach()
    .cpu()
    .clone()
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20068.44it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
train_dataset = PretrainedAnswerabilityDataset(df_train_tokenised, tokenizer)
val_dataset = PretrainedAnswerabilityDataset(df_val_tokenised, tokenizer)

collate_fn = make_collate_fn(tokenizer.pad_token_id)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn,
)

In [15]:
model = PretrainedGRUClassifier(
    pretrained_embeddings=pretrained_embeddings,
    hidden_dim=64,
    dropout_prob=0.3,
    padding_idx=tokenizer.pad_token_id,
).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    filter(
        lambda p: p.requires_grad,
        model.parameters(),
    ),
    lr=1e-3,
    weight_decay=1e-4,
)

In [16]:
NUM_EPOCHS = 10
PATIENCE = 3

best_val_f1 = -1
best_model_state = None
best_epoch = None

epochs_without_improvement = 0

for epoch in range(NUM_EPOCHS):

    train_loss = (
        train_one_epoch_answerability(
            model,
            train_loader,
            optimizer,
            criterion,
            device,
        )
    )

    val_f1 = evaluate_macro_f1(model, val_loader, device)

    if val_f1 > best_val_f1:

        best_val_f1 = val_f1
        best_epoch = epoch + 1

        best_model_state = {
            key: value.detach().cpu().clone() for key, value in model.state_dict().items()
        }
        epochs_without_improvement = 0

    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val Macro F1: {val_f1:.4f}"
    )

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping")
        break

Epoch 1/10 | Train loss: 0.1691 | Val Macro F1: 0.6986
Epoch 2/10 | Train loss: 0.0930 | Val Macro F1: 0.6986
Epoch 3/10 | Train loss: 0.0773 | Val Macro F1: 0.8046
Epoch 4/10 | Train loss: 0.0764 | Val Macro F1: 0.8484
Epoch 5/10 | Train loss: 0.0700 | Val Macro F1: 0.8612
Epoch 6/10 | Train loss: 0.0609 | Val Macro F1: 0.8023
Epoch 7/10 | Train loss: 0.0503 | Val Macro F1: 0.8137
Epoch 8/10 | Train loss: 0.0420 | Val Macro F1: 0.8252
Early stopping


In [17]:
print(f"Best epoch: {best_epoch}")
print(f"Best validation Macro F1: {best_val_f1:.4f}")

Best epoch: 5
Best validation Macro F1: 0.8612


In [18]:
model.load_state_dict(best_model_state)
model.to(device)

PretrainedGRUClassifier(
  (embeddings): Embedding(250002, 768, padding_idx=1)
  (gru): GRU(768, 64, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=256, out_features=1, bias=True)
)

In [ ]:
gru_pred, gru_prob, y_val, val_langs = predict_answerability(model, val_loader, device)
gru_results = evaluate_answerability_predictions(y_val, gru_pred, val_langs, "Pretrained GRU").round(4)
gru_results

,Model,Language,N,Macro F1,Unanswerable Precision,Unanswerable Recall,Unanswerable F1,Answerable Precision,Answerable Recall,Answerable F1
0,Pretrained GRU,Overall,1155,0.861163,0.879032,0.664634,0.756944,0.946654,0.984864,0.965381
1,Pretrained GRU,Arabic,415,0.951720,0.890909,0.942308,0.915888,0.991667,0.983471,0.987552
2,Pretrained GRU,Korean,356,0.900220,0.833333,0.789474,0.810811,0.988166,0.991098,0.989630
3,Pretrained GRU,Telugu,384,0.769231,0.882353,0.483871,0.625000,0.855856,0.979381,0.913462


In [21]:
all_results = pd.concat([majority_results, lr_results, gru_results], ignore_index=True).round(4)
all_results

,Model,Language,N,Macro F1,Unanswerable Precision,Unanswerable Recall,Unanswerable F1,Answerable Precision,Answerable Recall,Answerable F1
0,Majority Baseline,Overall,1155,0.4618,0.0000,0.0000,0.0000,0.8580,1.0000,0.9236
1,Majority Baseline,Arabic,415,0.4666,0.0000,0.0000,0.0000,0.8747,1.0000,0.9332
2,Majority Baseline,Korean,356,0.4863,0.0000,0.0000,0.0000,0.9466,1.0000,0.9726
3,Majority Baseline,Telugu,384,0.4311,0.0000,0.0000,0.0000,0.7578,1.0000,0.8622
4,Logistic Regression,Overall,1155,0.7236,0.8108,0.3659,0.5042,0.9038,0.9859,0.9431
5,Logistic Regression,Arabic,415,0.9459,0.8889,0.9231,0.9057,0.9889,0.9835,0.9862
6,Logistic Regression,Korean,356,0.4863,0.0000,0.0000,0.0000,0.9466,1.0000,0.9726
7,Logistic Regression,Telugu,384,0.5383,0.6000,0.1290,0.2124,0.7775,0.9725,0.8641
8,Pretrained GRU,Overall,1155,0.8612,0.8790,0.6646,0.7569,0.9467,0.9849,0.9654
9,Pretrained GRU,Arabic,415,0.9517,0.8909,0.9423,0.9159,0.9917,0.9835,0.9876
